# Catcher LLM — LangSmith 평가 실습

이 노트북은 셀을 **위에서 아래로 하나씩** 실행하면서 확인합니다.

| 단계 | 내용 |
|---|---|
| 0 | 환경 설정 & LangSmith 연결 확인 |
| 1 | Dataset — 구조 이해 & 생성 |
| 2 | Target 함수 — 평가할 대상 정의 |
| 3 | Evaluator ① — Heuristic (규칙 기반) |
| 4 | Evaluator ② — LLM-as-a-Judge (GPT가 채점) |
| 5 | evaluate() — 실험 실행 & 결과 확인 |
| 6 | A/B 테스트 — 프롬프트 두 개 비교 |
| 7 | 실패 케이스 찾기 |

---
## 0단계 — 환경 설정

`.env` 파일에서 API 키를 불러오고, LangSmith 서버에 실제로 연결되는지 확인합니다.

In [ ]:
# 셀 0-1: 경로 설정 + .env 로드
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# 프로젝트 루트 자동 탐색
def find_project_root() -> Path:
    curr = Path.cwd()
    for parent in [curr] + list(curr.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    return curr

PROJECT_ROOT = find_project_root()
src_dir = str(PROJECT_ROOT / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

load_dotenv(PROJECT_ROOT / ".env", override=True)

print("✅ 프로젝트 루트:", PROJECT_ROOT)

In [ ]:
# 셀 0-2: API 키 확인 (값은 숨기고 설정 여부만 표시)
# ❌ 가 뜨면 .env 파일에 해당 키를 추가해야 합니다
keys = [
    "LANGSMITH_API_KEY",
    "OPENAI_API_KEY",
    "LANGSMITH_PROJECT",
    "LANGSMITH_TRACING",
]

for key in keys:
    val = os.getenv(key, "")
    status = "✅ 설정됨" if val.strip() else "❌ 비어있음"
    print(f"{key:30s} {status}")

In [ ]:
# 셀 0-3: LangSmith 서버 연결 확인
# → 여기서 에러 나면 LANGSMITH_API_KEY 확인!
from langsmith import Client

client = Client(
    api_key=os.getenv("LANGSMITH_API_KEY"),
    api_url=os.getenv("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com"),
)

projects = list(client.list_projects(limit=5))
print("✅ LangSmith 연결 성공!")
print()
print("내 프로젝트 목록:")
for p in projects:
    print(f"  - {p.name}")

---
## 1단계 — Dataset

LangSmith Dataset = **평가용 문제지**입니다.

```
Dataset
├── Example 1: { inputs: {질문}, outputs: {정답} }
├── Example 2: { inputs: {질문}, outputs: {정답} }
└── Example 3: ...
```

- `inputs` → 모델에게 줄 입력
- `outputs` → 정답 (evaluator가 비교할 기준)

> ⚠️ Dataset은 한 번 만들면 LangSmith 서버에 저장됩니다. 이름이 같으면 기존 것을 재사용합니다.

In [ ]:
# 셀 1-1: 평가 예제 정의
# Catcher LLM 복지 정책 RAG용 QA 쌍

EXAMPLES = [
    {
        "inputs":  {"question": "여성청소년 생리용품 지원의 월 지원금은 얼마인가?"},
        "outputs": {"answer": "여성청소년 생리용품 지원의 월 지원금은 1만 4,000원이다."},
    },
    {
        "inputs":  {"question": "임신 사전건강관리 지원사업에서 여성에게 지원하는 최대 금액은?"},
        "outputs": {"answer": "임신 사전건강관리 지원사업에서 여성에게 지원하는 최대 금액은 13만 원이다."},
    },
    {
        "inputs":  {"question": "저소득 청소년부모 아동양육비 지원의 월 지원금은 얼마인가?"},
        "outputs": {"answer": "저소득 청소년부모 아동양육비 지원의 월 지원금은 25만 원이다."},
    },
    {
        "inputs":  {"question": "국·공립유치원 교육비는 월 얼마인가?"},
        "outputs": {"answer": "국·공립유치원 교육비는 월 10만 원이다."},
    },
    {
        "inputs":  {"question": "청년내일저축계좌의 정부 매칭 한도는 얼마인가?"},
        "outputs": {"answer": "청년내일저축계좌의 정부 매칭 한도는 월 최대 30만 원이다."},
    },
]

print(f"예제 {len(EXAMPLES)}개 준비됨")
print()
print("예제 구조 예시:")
print("  inputs  →", EXAMPLES[0]["inputs"])
print("  outputs →", EXAMPLES[0]["outputs"])

In [ ]:
# 셀 1-2: LangSmith에 Dataset 생성 (이미 있으면 기존 것 사용)
from langsmith.utils import LangSmithNotFoundError

DATASET_NAME = "catcher-llm-rag-eval"

try:
    dataset = client.read_dataset(dataset_name=DATASET_NAME)
    print(f"📂 기존 데이터셋 사용: '{DATASET_NAME}'")
except LangSmithNotFoundError:
    dataset = client.create_dataset(
        dataset_name=DATASET_NAME,
        description="Catcher LLM 복지 정책 RAG 평가용 데이터셋",
    )
    print(f"🆕 새 데이터셋 생성: '{DATASET_NAME}'")

# 예제가 없을 때만 업로드
existing = list(client.list_examples(dataset_id=dataset.id, limit=1))
if not existing:
    client.create_examples(dataset_id=dataset.id, examples=EXAMPLES)
    print(f"📤 예제 {len(EXAMPLES)}개 업로드 완료")
else:
    print("예제가 이미 있어서 업로드 건너뜀")

print()
print(f"Dataset ID: {dataset.id}")

In [ ]:
# 셀 1-3: Dataset에 들어있는 예제 목록 확인
# → LangSmith 대시보드의 Examples 탭과 같은 내용
print(f"=== '{DATASET_NAME}' 예제 목록 ===")
print()
for i, ex in enumerate(client.list_examples(dataset_id=dataset.id), 1):
    q = ex.inputs.get("question", "")
    a = (ex.outputs or {}).get("answer", "")
    print(f"[{i}] Q: {q}")
    print(f"     A: {a}")
    print()

---
## 2단계 — Target 함수

Target = **평가받을 모델/함수**입니다.

```
target(inputs) → outputs

inputs  = Dataset의 inputs 한 행 (예: {"question": "..."})
outputs = 모델이 생성한 결과  (예: {"answer": "...", "sources": [...]})
```

In [ ]:
# 셀 2-1: 실제 RAG target 함수 (프로젝트 기존 코드 사용)
from catcher_llm.config.settings import configure_langsmith_env, get_settings
from catcher_llm.services.rag_service import rag_target

settings = get_settings()
configure_langsmith_env(settings)

print("✅ RAG target 로드 완료")
print(f"   LLM: {settings.chat_model_label}")
print(f"   Embedding: {settings.embedding_model_label}")
print(f"   LangSmith 프로젝트: {settings.langsmith_project}")

In [ ]:
# 셀 2-2: target 함수 단독 테스트
# → evaluate() 돌리기 전에 먼저 답변이 나오는지 확인합니다
# → 여기서 에러 나면 vectorstore or LLM 설정 문제

test_input = {"question": "국·공립유치원 교육비는 월 얼마인가?"}

result = rag_target(test_input, settings=settings)

print("[입력]")
print(" ", test_input["question"])
print()
print("[답변]")
print(" ", result["answer"])
print()
print("[출처]")
for s in result["sources"]:
    print(" ", s)
print()
if result.get("error"):
    print("⚠️  에러:", result["error"])

---
## 3단계 — Evaluator ① Heuristic (규칙 기반)

LLM 없이 **코드 규칙만으로** 빠르게 채점하는 evaluator입니다.

```
장점: 빠르고 API 비용 없음
단점: 세밀한 의미 판단 불가
용도: 1차 필터, 구조 검사
```

In [ ]:
# 셀 3-1: Heuristic evaluator 3개 정의
import re
from langsmith.evaluation import EvaluationResult

def has_sources(inputs, outputs):
    """출처(sources)가 1개 이상 있는지 확인. score=1 좋음 / score=0 나쁨"""
    sources = outputs.get("sources", [])
    return EvaluationResult(key="has_sources", score=1 if sources else 0)

def contains_amount(inputs, outputs):
    """'14,000원', '30만 원' 같은 금액 표현이 있는지 확인."""
    answer = outputs.get("answer", "")
    has = bool(re.search(r"\d[\d,]*\s*만?\s*원", answer))
    return EvaluationResult(key="contains_amount", score=1 if has else 0)

FORBIDDEN_PHRASES = [
    "절약이 필요합니다", "소비를 줄여야", "돈을 아껴야", "재정 관리가 중요",
]

def no_generic_advice(inputs, outputs):
    """'절약하세요' 같은 일반론 표현이 없는지 확인. score=1 좋음 / score=0 나쁨"""
    answer = outputs.get("answer", "")
    found = any(phrase in answer for phrase in FORBIDDEN_PHRASES)
    return EvaluationResult(key="no_generic_advice", score=0 if found else 1)

print("✅ Heuristic evaluator 3개 정의 완료")
print("   - has_sources       : 출처 있는가?")
print("   - contains_amount   : 금액 표현 있는가?")
print("   - no_generic_advice : 일반론 표현 없는가?")

In [ ]:
# 셀 3-2: evaluator 단독 테스트
# → evaluate() 없이 직접 호출해서 채점 결과 확인

mock_good = {
    "answer": "국·공립유치원 교육비는 월 10만 원이다.",
    "sources": ["welfare/2026_hope_ladder_selected.pdf"],
}
mock_bad = {
    "answer": "재정 관리가 중요합니다.",
    "sources": [],
}

print("=== 좋은 답변 채점 ===")
for ev in [has_sources, contains_amount, no_generic_advice]:
    r = ev({}, mock_good)
    print(f"  {'✅' if r.score == 1 else '❌'} {r.key}: {r.score}")

print()
print("=== 나쁜 답변 채점 ===")
for ev in [has_sources, contains_amount, no_generic_advice]:
    r = ev({}, mock_bad)
    print(f"  {'✅' if r.score == 1 else '❌'} {r.key}: {r.score}")

---
## 4단계 — Evaluator ② LLM-as-a-Judge

GPT에게 "이 답변이 정답과 얼마나 일치하는지 점수 매겨줘" 라고 시키는 방식입니다.

```
장점: 의미 기반 채점 (단어가 달라도 뜻이 같으면 높은 점수)
단점: API 비용 발생, 느림
용도: groundedness, faithfulness 같은 세밀한 품질 측정
```

In [ ]:
# 셀 4-1: LLM-as-a-Judge evaluator 2개 정의
from langchain_openai import ChatOpenAI

judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def correctness_judge(inputs, outputs, reference_outputs):
    """모델 답변이 정답(reference)과 얼마나 일치하는지 GPT가 0~1 점수로 판단."""
    answer    = outputs.get("answer", "")
    reference = (reference_outputs or {}).get("answer", "")

    prompt = f"""다음 두 문장이 같은 내용인지 0.0~1.0 사이 숫자로만 평가해줘.
1.0 = 완전히 같은 의미, 0.0 = 전혀 다른 내용

정답: {reference}
모델 답변: {answer}

숫자 하나만 출력. 예: 0.8"""

    try:
        score = float(judge_llm.invoke(prompt).content.strip())
    except Exception:
        score = 0.0
    return EvaluationResult(key="correctness", score=score)


def faithfulness_judge(inputs, outputs, reference_outputs):
    """모델이 검색된 문서(contexts) 안에서만 답했는지 GPT가 판단."""
    answer   = outputs.get("answer", "")
    contexts = outputs.get("contexts", [])

    if contexts and isinstance(contexts[0], dict):
        context_text = "\n".join(c.get("content", "") for c in contexts)
    else:
        context_text = "\n".join(str(c) for c in contexts)

    if not context_text.strip():
        return EvaluationResult(key="faithfulness", score=0.0)

    prompt = f"""아래 답변이 오직 [문서] 안의 내용만 사용해서 작성됐는지 평가해줘.
문서 밖의 일반 지식이 포함됐으면 낮은 점수.
0.0~1.0 사이 숫자 하나만 출력.

[문서]
{context_text[:1000]}

[답변]
{answer}

숫자 하나만 출력. 예: 0.9"""

    try:
        score = float(judge_llm.invoke(prompt).content.strip())
    except Exception:
        score = 0.0
    return EvaluationResult(key="faithfulness", score=score)


print("✅ LLM-as-a-Judge evaluator 2개 정의 완료")
print("   - correctness_judge  : GPT가 정답 일치도 채점")
print("   - faithfulness_judge : GPT가 문서 충실성 채점")

In [ ]:
# 셀 4-2: judge evaluator 단독 테스트
# → API 호출 1번 일어남 (gpt-4o-mini 기준 약 0.01원)

test_outputs = {
    "answer": "국·공립유치원 교육비는 매월 10만 원입니다.",  # 표현이 달라도 의미는 같음
    "contexts": [{"content": "국·공립유치원 교육비(월) 10만 원, 사립유치원 28만 원"}],
}
test_ref = {"answer": "국·공립유치원 교육비는 월 10만 원이다."}

print("=== correctness_judge 테스트 ===")
r = correctness_judge({}, test_outputs, test_ref)
print(f"  점수: {r.score}  (1.0에 가까울수록 정답과 일치)")

print()
print("=== faithfulness_judge 테스트 ===")
r = faithfulness_judge({}, test_outputs, test_ref)
print(f"  점수: {r.score}  (1.0에 가까울수록 문서 기반으로만 답변)")

---
## 5단계 — evaluate() 실험 실행

`evaluate()` 가 하는 일:

```
Dataset의 각 예제
    → target 함수 호출 (RAG 답변 생성)
    → evaluator 5개가 채점
    → LangSmith 서버에 결과 저장
    → 터미널에 대시보드 URL 출력
```

In [ ]:
# 셀 5-1: evaluate() 실행
# → 실행 시간: 약 30~60초 (RAG + judge LLM 포함)
# → 완료되면 LangSmith URL이 자동 출력됩니다

from langsmith import evaluate

results = evaluate(
    lambda inputs: rag_target(inputs, settings=settings),
    data=DATASET_NAME,
    evaluators=[
        has_sources,
        contains_amount,
        no_generic_advice,
        correctness_judge,
        faithfulness_judge,
    ],
    experiment_prefix="rag-v1",   # ← A/B할 때 여기만 바꾸면 됨
    description="RAG 평가 첫 번째 실험 — 기본 프롬프트",
    max_concurrency=2,
    client=client,
)

print("✅ 실험 완료!")

In [ ]:
# 셀 5-2: 결과 요약 출력
from collections import defaultdict

scores = defaultdict(list)
for r in results:
    for fb in r.get("evaluation_results", {}).get("results", []):
        key   = getattr(fb, "key", None)
        score = getattr(fb, "score", None)
        if key and score is not None:
            scores[key].append(float(score))

PASS_THRESHOLD = 0.7

print("=" * 50)
print("평가 결과 요약 (rag-v1)")
print("=" * 50)
print(f"{'항목':<25} {'평균':>6}  {'pass/total':>12}")
print("-" * 50)

for key, vals in sorted(scores.items()):
    avg    = sum(vals) / len(vals)
    passes = sum(1 for v in vals if v >= PASS_THRESHOLD)
    emoji  = "✅" if avg >= PASS_THRESHOLD else "❌"
    print(f"{emoji} {key:<23} {avg:>6.2f}  {passes}/{len(vals)}")

print()
print("LangSmith 대시보드:")
print("  smith.langchain.com → Datasets → catcher-llm-rag-eval → Experiments")

---
## 6단계 — A/B 테스트

프롬프트 A와 B를 **같은 Dataset**에 돌려서 점수를 비교합니다.

```
experiment_prefix="rag-v1"  → 기존 프롬프트 (5단계에서 이미 실행)
experiment_prefix="rag-v2"  → 아래에서 개선한 프롬프트로 실행

LangSmith → Experiments → 두 개 체크박스 선택 → Compare
```

In [ ]:
# 셀 6-1: v2 실험 실행 (experiment_prefix만 바꿈)
# 실제로는 프롬프트나 모델을 바꾼 뒤 실행합니다

results_v2 = evaluate(
    lambda inputs: rag_target(inputs, settings=settings),
    data=DATASET_NAME,
    evaluators=[
        has_sources,
        contains_amount,
        no_generic_advice,
        correctness_judge,
        faithfulness_judge,
    ],
    experiment_prefix="rag-v2",   # ← 여기만 변경
    description="RAG 평가 두 번째 실험",
    max_concurrency=2,
    client=client,
)

print("✅ v2 실험 완료!")

In [ ]:
# 셀 6-2: v1 vs v2 점수 직접 비교

def summarize(results_obj):
    s = defaultdict(list)
    for r in results_obj:
        for fb in r.get("evaluation_results", {}).get("results", []):
            key   = getattr(fb, "key", None)
            score = getattr(fb, "score", None)
            if key and score is not None:
                s[key].append(float(score))
    return {k: sum(v)/len(v) for k, v in s.items()}

v1 = summarize(results)
v2 = summarize(results_v2)
all_keys = sorted(set(v1) | set(v2))

print(f"{'항목':<25} {'v1':>6}  {'v2':>6}  {'변화':>8}")
print("-" * 52)
for key in all_keys:
    s1   = v1.get(key, 0)
    s2   = v2.get(key, 0)
    diff = s2 - s1
    arrow = "⬆️" if diff > 0.05 else ("⬇️" if diff < -0.05 else "➡️")
    print(f"  {key:<23} {s1:>6.2f}  {s2:>6.2f}  {arrow} {diff:>+.2f}")

---
## 7단계 — 실패 케이스 분석

In [ ]:
# 셀 7-1: 실패 케이스 추출 (score < 0.7 인 행)

FAIL_THRESHOLD = 0.7
fail_count = 0

print(f"=== 실패 케이스 분석 (기준: score < {FAIL_THRESHOLD}) ===")
print()

for i, r in enumerate(results, 1):
    question = r.get("inputs", {}).get("question", "")
    answer   = r.get("outputs", {}).get("answer", "")
    failed   = [
        (getattr(fb, "key", ""), getattr(fb, "score", 1))
        for fb in r.get("evaluation_results", {}).get("results", [])
        if getattr(fb, "score", 1) is not None
        and float(getattr(fb, "score", 1)) < FAIL_THRESHOLD
    ]
    if failed:
        fail_count += 1
        print(f"[케이스 {i}] ❌")
        print(f"  Q: {question}")
        print(f"  A: {answer[:80]}..." if len(answer) > 80 else f"  A: {answer}")
        for key, score in failed:
            print(f"  실패 항목: {key} = {float(score):.2f}")
        print()

print("🎉 모든 케이스 통과!" if fail_count == 0 else f"총 {fail_count}개 실패")

In [ ]:
# 셀 7-2: 실패 케이스에 LangSmith 태그 붙이기
# → LangSmith에서 failure_type 으로 필터해서 실패 유형별 분류 가능

FAILURE_TAXONOMY = {
    "faithfulness":      "F-04_rag_out_of_context",
    "correctness":       "F-01_hallucination_or_wrong",
    "has_sources":       "F-04_no_source_returned",
    "contains_amount":   "F-01_no_specific_amount",
    "no_generic_advice": "F-02_generic_advice",
}

tagged = 0
for r in results:
    run_id = r.get("run") and getattr(r["run"], "id", None)
    if not run_id:
        continue
    for fb in r.get("evaluation_results", {}).get("results", []):
        key   = getattr(fb, "key", "")
        score = getattr(fb, "score", 1)
        if score is not None and float(score) < FAIL_THRESHOLD and key in FAILURE_TAXONOMY:
            client.create_feedback(
                run_id=run_id,
                key="failure_type",
                value=FAILURE_TAXONOMY[key],
                comment=f"{key} score={float(score):.2f}",
            )
            tagged += 1

print(f"✅ 실패 태그 {tagged}개 부착 완료")
print()
print("LangSmith 확인:")
print("  Projects → catcher-llm → Runs → Filter: feedback.failure_type is not null")

---
## 전체 흐름 정리

```
0단계  환경 설정     → API 키 로드 + LangSmith 연결
1단계  Dataset       → QA 쌍 만들어서 LangSmith에 업로드
2단계  Target        → 평가받을 RAG 함수 정의 & 단독 테스트
3단계  Evaluator ①  → 규칙 기반 채점 (빠르고 무료)
4단계  Evaluator ②  → GPT 의미 기반 채점 (정확, 비용 발생)
5단계  evaluate()    → 실험 실행 + LangSmith 결과 저장
6단계  A/B 테스트   → experiment_prefix 바꿔서 비교
7단계  실패 분석    → 낮은 점수 케이스 찾고 태그
```

### 다음에 바꿔볼 수 있는 것
- `experiment_prefix` → `"rag-v3"`, `"rag-chunk512"` 등으로 다양한 설정 비교
- `evaluators` 리스트 → judge prompt 교체해서 다른 기준으로 채점
- Dataset 예제 추가 → `client.create_example(...)` 후 재실험
- `chunk_size`, `top_k` 변경 → retrieval 성능 비교